<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week6/lesson21/Lesson21_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 21 — Exercises**
## ODEs and the Passive Membrane: Equilibria, Time Constants and the Euler Method

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 6 — Differential Equations\
**Lesson:** Lesson 21

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Relaxation and time constants | 15 |
| 2 | Equilibria, stability and the phase line | 15 |
| 3 | The membrane equation and its units | 15 |
| 4 | The Euler method: accuracy and stability | 15 |
| 5 | Non-autonomous input and a reference solver | 15 |
| 6 | Superposition and its failure | 10 |
| 7 | Real data: is this neuron passive? | 15 |
| **Total** | | **100** |
| Bonus | Heun's method (second order) | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit, brentq

PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

print("Setup complete.")

---
## Exercise 1 — Relaxation and Time Constants (15 points)

After a burst of spikes, the calcium concentration in a dendrite decays back to baseline: $\tau_{Ca}\,dc/dt = -(c - c_\infty)$ with $c_\infty = 0.05$ µM and $\tau_{Ca} = 80$ ms.

In [ ]:
# 1.1 (6 points): Write relax(t, x0, x_inf, tau) returning the exact solution x_inf + (x0 - x_inf) exp(-t / tau).
# (a) Check it solves the ODE: with t1 = np.linspace(0, 400, 4001) (ms) and c0 = 0.5 uM, store in ode_err1 the maximum of
#     |np.gradient(c, t1) + (c - 0.05) / 80| (uM/ms).
# (b) Store t90_1 and t99_1: the times (ms) at which 90% and 99% of the decay from 0.5 to 0.05 uM is complete (use the formula, not the grid).
# (c) Store t_below1: the time (ms) at which c falls below 0.1 uM.

t1 = np.linspace(0, 400, 4001)                                              # (4001,) ms

# YOUR CODE HERE
ode_err1 = t90_1 = t99_1 = t_below1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.1
assert all(v is not None for v in (ode_err1, t90_1, t99_1, t_below1)), "Set ode_err1, t90_1, t99_1 and t_below1."
assert ode_err1 < 1e-4, "The exact solution satisfies the ODE up to finite-difference error."
assert np.isclose(t90_1, 184.2, atol=0.1) and np.isclose(t99_1, 2 * t90_1), "Fraction remaining e^{-t/tau}: 0.1 at tau ln 10."
assert np.isclose(relax(t_below1, 0.5, 0.05, 80.0), 0.1), "At t_below1 the concentration is exactly 0.1 uM."
print("Self-check passed.")

In [ ]:
# 1.2 (5 points): Estimating a time constant from noisy data. ca_obs below is a noisy decay sampled every 5 ms.
# (a) Log-linear method: assuming c_inf = 0.05 is known, fit a straight line to log(ca_obs - 0.05) against t, using only samples with
#     ca_obs - 0.05 > 0.02 (the log of small or negative numbers is useless). Store tau_log1 = -1 / slope.
# (b) Nonlinear least squares: fit relax(t, x0, x_inf, tau) to ALL samples with curve_fit (p0 = [0.5, 0.05, 50]). Store tau_nls1 and xinf_nls1.

rng1 = np.random.default_rng(1)
t_obs1 = np.arange(0, 500, 5.0)                                             # (100,) ms
ca_obs = 0.05 + 0.45 * np.exp(-t_obs1 / 80.0) + rng1.normal(0, 0.01, t_obs1.size)   # (100,) uM, noise SD 0.01 uM

# YOUR CODE HERE
tau_log1 = tau_nls1 = xinf_nls1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert all(v is not None for v in (tau_log1, tau_nls1, xinf_nls1)), "Set tau_log1, tau_nls1 and xinf_nls1."
assert abs(tau_nls1 - 80) < 4 and abs(xinf_nls1 - 0.05) < 0.01, "The nonlinear fit recovers tau and the baseline."
assert abs(tau_log1 - 80) < 10, "The log-linear fit is roughly right."
print("Self-check passed.")

**1.3 (4 points, written response).** (a) Why does the log-linear method need $c_\infty$ to be known, and why must samples close to $c_\infty$ be dropped? (b) Is the calcium equation autonomous? Would it still be if calcium entered through a time-varying current $j(t)$? Write that equation. (c) "The decay is over after one time constant." Correct this statement.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Equilibria, Stability and the Phase Line (15 points)

A population with recurrent excitation obeys $\tau\,dr/dt = g(r) = -r + F(wr + I)$, with $F(u) = 1/(1 + e^{-(u - 0.5)/0.08})$ and $\tau = 10$ ms; $r$ is the rate as a fraction of the maximum.

In [ ]:
# 2.1 (6 points): Write equilibria(w, I) returning a list of (r_star, 'stable' or 'unstable') for r in [0, 1]: find sign changes of g on a fine
# grid (np.linspace(0, 1, 2000)), refine each with brentq, and classify by the sign of g'(r_star) (a centred finite difference, h = 1e-6).
# Store eq_list2 = {w: equilibria(w, 0.0) for w in [0.6, 1.0, 1.4]}, and n_eq2 = {w: number of equilibria}.

def F2(u):
    return 1 / (1 + np.exp(-(u - 0.5) / 0.08))
def g2(r, w, I):
    return -r + F2(w * r + I)

# YOUR CODE HERE
eq_list2 = n_eq2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.1
assert eq_list2 is not None and n_eq2 is not None, "Set eq_list2 and n_eq2."
assert n_eq2[0.6] == 1 and n_eq2[1.0] == 3 and n_eq2[1.4] == 3, "Weak recurrence: one equilibrium; strong: three."
assert [s for _, s in eq_list2[1.0]] == ['stable', 'unstable', 'stable'], "Stable, unstable, stable from low to high r."
assert all(abs(g2(rs, 1.4, 0.0)) < 1e-9 for rs, _ in eq_list2[1.4]), "Each r* is a zero of g."
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): Memory in a bistable unit. With w = 1.4 and tau = 10 ms, simulate with solve_ivp from r0 in r0_list2 for 200 ms with I = 0, and store
# r_final2 (shape (5,)). Then apply a brief input: starting from the LOW state (r0 = 0), set I = 0.6 for 20 ms and then I = 0 for 180 ms; store
# r_after2, the rate at 200 ms.

r0_list2 = [0.0, 0.2, 0.35, 0.45, 0.8]

# YOUR CODE HERE
r_final2 = r_after2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert r_final2 is not None and r_after2 is not None, "Set r_final2 and r_after2."
r_mid = eq_list2[1.4][1][0]
assert all((rf < 0.1) == (r0 < r_mid) for rf, r0 in zip(r_final2, r0_list2)), "Below the unstable point the rate falls to the low state, above it rises to the high state."
assert r_after2 > 0.9, "A brief input switches the unit to the high state, where it stays."
print("Self-check passed.")

**2.3 (4 points, written response).** (a) Explain, using the phase line, why the unstable equilibrium in 2.2 acts as a threshold between two outcomes. (b) Why could the passive membrane equation never show the memory of 2.2? (c) Name one neural or machine-learning system that uses this kind of bistability.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — The Membrane Equation and Its Units (15 points)

$\tau_m\,dV/dt = -(V - E_L) + R_m I(t)$ with $\tau_m = R_m C_m$. Work in ms, mV, pA, MΩ and pF, and remember MΩ × pF = µs and MΩ × pA = µV.

In [ ]:
# 3.1 (5 points): Write passive_params(R_Mohm, C_pF, I_pA) returning (tau_ms, dV_mV): the membrane time constant in ms and the steady
# deflection in mV. Store results3 = [passive_params(*row) for row in cells3] and t95_3 = the time (ms) for each cell to come within 5% of
# its steady deflection after a step (list of 3).

cells3 = [(250, 80, 40), (90, 250, 200), (500, 30, 10)]                     # (R megaohm, C pF, I pA): a small, a large, a tiny neuron

# YOUR CODE HERE
results3 = t95_3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert results3 is not None and t95_3 is not None, "Set results3 and t95_3."
assert np.allclose(results3[0], (20.0, 10.0)) and np.allclose(results3[1], (22.5, 18.0)) and np.allclose(results3[2], (15.0, 5.0)), "Check the unit conversions."
assert np.allclose(t95_3, [3.0 * r[0] for r in results3], rtol=0.01), "5% remains after about 3 tau (tau ln 20)."
print("Self-check passed.")

In [ ]:
# 3.2 (6 points): Temporal summation. For tau_m = 20 ms, R_m = 200 megaohm (0.2 mV/pA), E_L = -70 mV, inject two 2 ms pulses of 500 pA, the second
# starting dt_sep ms after the first. Using the exact update for piecewise-constant input on a 0.01 ms grid, store peak3 (shape (6,)): the peak
# deflection (mV above rest) for each dt_sep in sep_list3, and peak_single3: the peak for one pulse alone.

sep_list3 = [2, 5, 10, 20, 40, 80]                                           # ms between pulse onsets
tg3 = np.arange(0, 300, 0.01)                                                # (30000,) ms

# YOUR CODE HERE
peak3 = peak_single3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert peak3 is not None and peak_single3 is not None, "Set peak3 and peak_single3."
assert np.all(np.diff(peak3) < 0) and peak3[0] > 1.8 * peak_single3 and peak3[-1] < 1.1 * peak_single3, "Summation fades as the separation grows."
pred = peak_single3 * (1 + np.exp(-np.array(sep_list3[2:]) / 20.0))
assert np.allclose(peak3[2:], pred, rtol=0.05), "For separated pulses the peak is about single x (1 + e^{-sep / tau})."
print("Self-check passed.")

**3.3 (4 points, written response).** (a) The "tiny neuron" of 3.1 has the shortest time constant but the largest resistance. Explain why its 10 pA input moves it by 5 mV while 200 pA moves the large neuron by 18 mV. (b) Explain the formula used in the 3.2 self-check, and what it means for a neuron receiving inputs at 20 Hz versus 200 Hz. (c) Why is $\tau_m$ independent of cell size for a uniform membrane, although $R_m$ and $C_m$ both depend on it?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — The Euler Method: Accuracy and Stability (15 points)

In [ ]:
# 4.1 (6 points): Write euler(f, x0, t) (forward Euler on the grid t). For dV/dt = (-(V + 70) + 0.2 * 150) / 15 (mV/ms; tau 15 ms, 150 pA), V(0) = -70,
# on [0, 90] ms, store err4 (shape (6,)): the maximum absolute error against the exact solution for each dt in dt_list4, and order4: the
# slope of log(err) against log(dt). Also store err1step4: the error after ONE step of size dt for each dt (shape (6,)), and its slope order1step4.

dt_list4 = [3.0, 1.5, 0.75, 0.375, 0.1875, 0.09375]                          # ms
f4 = lambda V, t: (-(V + 70) + 0.2 * 150) / 15
exact4 = lambda t: -70 + 30 * (1 - np.exp(-t / 15))

# YOUR CODE HERE
err4 = order4 = err1step4 = order1step4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert all(v is not None for v in (err4, order4, err1step4, order1step4)), "Set err4, order4, err1step4 and order1step4."
assert abs(order4 - 1) < 0.1 and abs(order1step4 - 2) < 0.1, "Global error O(dt), local error O(dt^2)."
assert err4[-1] < 0.1, "With dt about 0.1 ms the error is below 0.1 mV."
print("Self-check passed.")

In [ ]:
# 4.2 (5 points): Stability. For dx/dt = -x / tau with tau = 10 ms, x(0) = 1, run Euler for 200 ms with each dt in dt_list4b. Store
# behaviour4 = {dt: 'monotone' / 'oscillating decay' / 'growing'} classified from the numerical solution (sign changes; |x| at the end vs 1),
# and xend_exp4 = {dt: final value} from the EXACT (exponential) update x[n+1] = x[n] * exp(-dt / tau) for the same steps.

dt_list4b = [5, 9, 12, 19, 21, 30]                                            # ms

# YOUR CODE HERE
behaviour4 = xend_exp4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert behaviour4 is not None and xend_exp4 is not None, "Set behaviour4 and xend_exp4."
assert behaviour4 == {5: 'monotone', 9: 'monotone', 12: 'oscillating decay', 19: 'oscillating decay', 21: 'growing', 30: 'growing'}, "Thresholds at dt = tau and dt = 2 tau."
assert all(0 < v < 1e-7 for v in xend_exp4.values()), "The exact update decays for any step size."
print("Self-check passed.")

**4.3 (4 points, written response).** (a) Explain why the one-step error is $O(\Delta t^2)$ but the error at a fixed time is $O(\Delta t)$. (b) Derive the stability condition $\Delta t < 2\tau$ for Euler on $dx/dt = -x/\tau$. (c) A network model contains synapses with $\tau = 2$ ms and membranes with $\tau = 20$ ms. What does this imply for the Euler step?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Non-Autonomous Input and a Reference Solver (15 points)

A sinusoidal current $I(t) = I_0 \sin(2\pi f t)$ drives a membrane with $\tau_m = 20$ ms, $R_m = 200$ MΩ. After the start-up transient the voltage oscillates at the same frequency with amplitude $R_m I_0 / \sqrt{1 + (2\pi f \tau_m)^2}$.

In [ ]:
# 5.1 (6 points): For each f in f_list5 (Hz), simulate 0-1500 ms with solve_ivp (rtol 1e-8, atol 1e-8, max_step 0.5 ms), I0 = 50 pA, V(0) = E_L = -70,
# and measure the steady amplitude as half the peak-to-peak of V over the last 500 ms (at least one full period). Store amp_sim5 and amp_theory5 (shape (5,), mV) and
# f_cut5 = 1 / (2 pi tau) in Hz.

f_list5 = [2, 4, 8, 20, 50]                                                  # Hz
tau5, R5, EL5, I05 = 20.0, 0.2, -70.0, 50.0                                  # ms, mV/pA, mV, pA

# YOUR CODE HERE
amp_sim5 = amp_theory5 = f_cut5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert all(v is not None for v in (amp_sim5, amp_theory5, f_cut5)), "Set amp_sim5, amp_theory5 and f_cut5."
assert np.allclose(amp_sim5, amp_theory5, rtol=0.01), "Simulated and theoretical amplitudes agree."
assert np.isclose(f_cut5, 7.96, atol=0.01) and np.isclose(amp_theory5[2] / (R5 * I05), 1 / np.sqrt(2), atol=0.01), "At the cutoff the amplitude falls by sqrt(2)."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Accuracy against cost. For f = 20 Hz on 0-200 ms, use as reference a solve_ivp run with rtol = atol = 1e-11 (method 'DOP853').
# (a) For each rtol in rtol_list5 (with atol = rtol * 1e-2, default RK45, no max_step), store the max error at t_cmp5 and the number of function
#     evaluations: err_ivp5, nfev5 (each shape (4,)).
# (b) For Euler with dt in dt_list5, store err_eul5 at the same points (nfev = number of steps).

rtol_list5 = [1e-2, 1e-4, 1e-6, 1e-8]
dt_list5 = [1.0, 0.1, 0.01]                                                   # ms
t_cmp5 = np.arange(0, 200.01, 1.0)                                            # (201,) ms

# YOUR CODE HERE
err_ivp5 = nfev5 = err_eul5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert all(v is not None for v in (err_ivp5, nfev5, err_eul5)), "Set err_ivp5, nfev5 and err_eul5."
assert err_ivp5[-1] < 1e-5 and nfev5[-1] < 2000 and np.all(np.diff(nfev5) > 0), "Tighter tolerance: smaller error, more evaluations."
assert np.allclose(err_eul5[:-1] / err_eul5[1:], 10, rtol=0.15), "Euler: ten times smaller dt, ten times smaller error."
assert err_ivp5[-1] < err_eul5[-1], "RK45 at rtol 1e-8 beats Euler with 20,000 steps."
print("Self-check passed.")

**5.3 (4 points, written response).** (a) In 5.1, why must the amplitude be measured after the first few hundred milliseconds? (b) The cutoff frequency of a neuron with $\tau_m = 20$ ms is about 8 Hz. Does this mean the neuron cannot respond to 40 Hz gamma-band input? (c) Why would you still choose Euler with a small fixed step over an adaptive solver when simulating a network of spiking neurons?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Superposition and Its Failure (10 points)

Linear ODEs obey **superposition**: the deviation from rest caused by the sum of two inputs is the sum of the deviations caused by each. We test it, then break it with a voltage-dependent conductance.

In [ ]:
# 6.1 (6 points): With tau = 20 ms, R = 0.2 mV/pA, E_L = -70 on the grid tg6 and the currents Ia6, Ib6 below:
# (a) Simulate the PASSIVE model with Euler for Ia6, Ib6 and Ia6 + Ib6; store sup_err_lin6 = max |(V_ab + 70) - ((V_a + 70) + (V_b + 70))|.
# (b) Add a depolarising conductance that opens with depolarisation (reversal +50 mV):
#         tau dV/dt = -(V - E_L) + R I + 0.08 * m(V) * (50 - V),   m(V) = 1 / (1 + exp(-(V + 60) / 4)).
#     Its resting potential is no longer exactly -70 mV: find it with brentq on [-80, -65] and start there. Using deviations from that rest,
#     store sup_err_nl6 = max |dV_ab - (dV_a + dV_b)| and peak_ratio6 = max(dV_ab) / max(dV_a + dV_b).

tg6 = np.arange(0, 300, 0.05)                                                # (6000,) ms
Ia6 = np.where((tg6 >= 20) & (tg6 < 140), 30.0, 0.0)                          # (6000,) pA
Ib6 = np.where((tg6 >= 80) & (tg6 < 200), 20.0, 0.0)                          # (6000,) pA

# YOUR CODE HERE
sup_err_lin6 = sup_err_nl6 = peak_ratio6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert all(v is not None for v in (sup_err_lin6, sup_err_nl6, peak_ratio6)), "Set sup_err_lin6, sup_err_nl6 and peak_ratio6."
assert sup_err_lin6 < 1e-9, "The passive model obeys superposition exactly (up to rounding)."
assert sup_err_nl6 > 1 and peak_ratio6 > 1.03, "The voltage-dependent conductance makes the combined response larger than the sum."
print("Self-check passed.")

**6.2 (4 points, written response).** (a) Why does superposition hold exactly for the passive model, even with Euler? (b) In the mini-lab neuron, depolarising steps produced larger deflections than $R_m I$. Connect this to 6.1(b). (c) Give one experimental test, using only current steps, that would show a neuron is not passive.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Is This Neuron Passive? (15 points)

**Data.** A second whole-cell current-clamp recording from the pyABF sample files (`171116sh_0018.abf`, MIT licence; cell type undocumented), reduced to 10 kHz: `patch_171116sh_0018_steps.npz`. 17 sweeps of 3 s.
- `V7` (17, 30000): membrane potential, mV. `I7` (17, 30000): injected current, pA. `t7` (30000,): time, s.
- Every sweep injects **−100 pA from 1.147 to 1.647 s**. In addition, sweep $k$ injects a step of $-100 + 25k$ pA from 0.147 to 0.647 s (sweeps 0–3 are hyperpolarising: −100, −75, −50, −25 pA).

In [ ]:
# PROVIDED: load the recording.
import os, urllib.request
DATA_FILE7 = 'patch_171116sh_0018_steps.npz'
for cand in [DATA_FILE7, os.path.join('..', '..', 'data', DATA_FILE7)]:
    if os.path.exists(cand):
        path7 = cand; break
else:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/peter-adepoju/math-neuro-ml/e8996ac071a2a16c48d5b544ec3470816eecf49f/data/' + DATA_FILE7, DATA_FILE7)
    path7 = DATA_FILE7
d7 = np.load(path7)
V7 = d7['v_cmV'] / 100.0                                                      # (17, 30000) mV
I7 = d7['i_pA'].astype(float)                                                 # (17, 30000) pA
fs7 = float(d7['fs_hz']); t7 = np.arange(V7.shape[1]) / fs7                   # (30000,) s
a7 = int(np.where(np.diff(I7[0, int(1.0 * fs7):]) != 0)[0][0] + 1 + int(1.0 * fs7))   # index where the shared -100 pA step starts
b7 = a7 + int(0.5 * fs7)                                                      # 500 ms later
print(V7.shape, f"shared step from {t7[a7]:.4f} s to {t7[b7]:.4f} s")

In [ ]:
# 7.1 (7 points): Fit the passive model to the shared -100 pA step.
# (a) Average the 17 sweeps; E_L7 = mean of the average over 1.00 s to the step onset.
# (b) Fit relax(t, v_inf, tau, v0) (note the argument order: use lambda tt, vinf, tau, v0: vinf + (v0 - vinf) * np.exp(-tt / tau)) to the first
#     W ms after onset for each W in win_list7, and store tau7 and R7 (each shape (4,); ms and megaohm, R = (v_inf - E_L7) / (-100 pA)).
# (c) Store sag7: the steady value (mean of the last 20 ms of the step) minus the minimum of the average trace during the step (mV, positive), and
#     sag_ratio7 = (steady - E_L7) / (minimum - E_L7).

win_list7 = [50, 100, 200, 500]                                               # ms

# YOUR CODE HERE
E_L7 = tau7 = R7 = sag7 = sag_ratio7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert all(v is not None for v in (E_L7, tau7, R7, sag7, sag_ratio7)), "Set E_L7, tau7, R7, sag7 and sag_ratio7."
assert tau7[0] > 1.6 * tau7[-1] and R7[0] > R7[-1], "Fits that include the sag give a shorter tau and smaller R."
assert 2.5 < sag7 < 4.5 and 0.65 < sag_ratio7 < 0.85, "A sag of a few millivolts: the cell is not passive."
print("Self-check passed.")

In [ ]:
# 7.2 (4 points): Linearity. For sweeps 0-3 (steps of -100, -75, -50, -25 pA from 0.147 to 0.647 s), compute for each sweep the baseline
# (mean over 50-140 ms), the peak deflection (minimum of the trace smoothed with a 5 ms moving average, during the step, minus the baseline),
# and the steady deflection (mean of the last 50 ms of the step minus the baseline). Store I_hyp7, peak_dV7, ss_dV7 (each shape (4,)) and the
# slopes (megaohm) of straight-line fits through the origin: R_peak7 and R_ss7.

# YOUR CODE HERE
I_hyp7 = peak_dV7 = ss_dV7 = R_peak7 = R_ss7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert all(v is not None for v in (I_hyp7, peak_dV7, ss_dV7, R_peak7, R_ss7)), "Set all five results."
assert np.all(peak_dV7 < ss_dV7) and np.all(ss_dV7 < 0), "Every hyperpolarising step shows a sag (peak beyond steady)."
assert R_peak7 > 1.1 * R_ss7, "Peak and steady 'resistances' differ."
print("Self-check passed.")

**7.3 (4 points, written response).** (a) Which of your time constants would you report as this neuron's membrane time constant, and why? (b) What does the sag tell you about the neuron, and why can no choice of $\tau_m$, $R_m$ and $E_L$ in the passive model reproduce it? (c) How would you extend the model by one equation to capture the sag? (Write the equation; you do not need to fit it.)

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Heun's Method (extra credit)

Heun's method (also called improved Euler, a second-order Runge–Kutta method) takes a trial Euler step and then averages the slopes at both ends:
$$\tilde x = x_n + \Delta t\,f(x_n, t_n), \qquad x_{n+1} = x_n + \tfrac{\Delta t}{2}\big(f(x_n, t_n) + f(\tilde x, t_{n+1})\big).$$
1. Implement it and repeat Exercise 4.1's error table. What is the order?
2. Compare the error of Euler and Heun at **equal cost** (Heun uses two evaluations of $f$ per step).
3. Is Heun stable for $\Delta t = 2.5\tau$ on $dx/dt = -x/\tau$? Derive its amplification factor.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, a legend where appropriate, and no background grid.
- Every simulated quantity is reported with its units (ms, mV, pA, megaohm, pF), and every simulation states its time step or solver tolerance.
- You checked at least one simulation by halving the time step or against an exact or reference solution.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.